In [44]:
import stanza
import re
import os
import json
from pathlib import Path
from nltk import sent_tokenize, word_tokenize
from tqdm import tqdm
import pandas as pd
import json
import requests

In [2]:

# Load the text file

corpus = dict()

folder_path = os.path.expanduser('~/Desktop/TXT_all')

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
            corpus[filename] = text  


## Named Entity Recognition

In [4]:

df_list = []

nlp = stanza.Pipeline(lang='en', processors='tokenize,ner')

for author in corpus:

    sentences = sent_tokenize(corpus[author])

    for sentence in tqdm(sentences):
        doc = nlp(sentence)
        entities = doc.ents
        for entity in entities:
            data = json.loads(str(entity))
            if data['type'] == 'LOC' or data['type'] == 'PERSON' or data['type'] == 'GPE':
                row = []
                row.append(data['type'])
                row.append(data['text'])
                author_name = re.sub('.txt','',author)
                row.append(author_name)
                df_list.append(row)


2025-09-27 11:50:49 INFO: Loading these models for language: en (English):
| Processor | Package   |
-------------------------
| tokenize  | combined  |
| ner       | ontonotes |

2025-09-27 11:50:49 INFO: Use device: cpu
2025-09-27 11:50:49 INFO: Loading: tokenize
2025-09-27 11:50:49 INFO: Loading: ner
2025-09-27 11:50:49 INFO: Done loading processors!
100%|███████████████████████████████████████| 1652/1652 [04:37<00:00,  5.94it/s]


In [27]:
df = pd.DataFrame(df_list,columns=['type','value','author'],index=None)

In [194]:
persons = df.query('type =="PERSON" ' )
locations = df.query('type == "LOC" or type == "GPE"' )

## Openrefine

In [298]:
with open('locations.csv','w',encoding='utf-8') as out:
    out.write('location\n')
    for loc in locations['value'].unique():
        out.write(f"{loc}\n")
        
with open('persons.csv','w',encoding='utf-8') as out:
    out.write('name\n')
    for loc in persons['value'].unique():
        out.write(f"{loc}\n")

## Create files for Gephi

In [299]:
openrefine = pd.read_csv('openrefine_location.csv')

keys = dict()

for i,row in openrefine.iterrows():
    if not(pd.isna(row['wikidata_countries'])):
        keys[row['location']] = row['wikidata_countries']
        
    if not(pd.isna(row['wikidata_city'])):
        keys[row['location']] = row['wikidata_city']
        

In [300]:

def wikidata_query(query):
    url = 'https://query.wikidata.org/sparql'
    try:
        r = requests.get(url, params = {'format': 'json', 'query': query})
        return r.json()['results']['bindings']
    except json.JSONDecodeError as e:
        raise Exception('Invalid query')

keys_list = list(keys.values())
keys_list = [f'wd:{key}' for key in keys_list]

wd_labels = dict()
    

item2label = wikidata_query('''
    SELECT ?item ?itemLabel WHERE {
    VALUES ?item { %s }
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}''' % " ".join(keys_list))

for result in item2label :
    item = re.sub(r".*[#/\\]", "", result['item']['value'])
    label = result['itemLabel']['value']
    wd_labels[item] = label
    


## Gephi Files

In [301]:
for i,author in enumerate(locations['author'].unique()):
    keys[author] = f"AUTH00{i+1}"
    wd_labels[f"AUTH00{i+1}"] = author

In [302]:
edges = []
nodes = []



for i,row in locations.iterrows():
    if row['value'] in keys:
        edges.append( [ keys[row['author']],keys[row['value']] ] )
        nodes.append( keys[row['author']] )
        nodes.append( keys[row['value']] )
        
nodes = list(set(nodes))


In [303]:
with open('locations_edges.csv','w') as out:
    out.write('Source,Target\n')
    for edge in edges:
        out.write( f"'{edge[1]}','{edge[0]}'\n")
        
        
with open('locations_nodes.csv','w') as out:
    out.write('Id,Label,Type\n')
    for node in nodes:
        if re.search(r'^AUTH',node):
            out.write(f"'{node}','{wd_labels[node]}','Author'\n")
        else:
            wd_labels[node]=re.sub(',','',str(wd_labels[node]))
            out.write(f"'{node}','{wd_labels[node]}','Location'\n")

In [304]:
keys

{'LisBoA': 'Q597',
 'Monrovia,': 'Q3748',
 'Brazil': 'Q2844',
 'Uruguay': 'Q77',
 'Napoles': 'Q2634',
 'Colombia': 'Q739',
 'Mexico': 'Q96',
 'Cuba': 'Q241',
 'La Haya': 'Q36600',
 'Portugal': 'Q45',
 'Chile': 'Q298',
 'Argentina': 'Q414',
 'N.Y.': 'Q60',
 'Nueva York': 'Q60',
 'WasuincTon D.C.': 'Q61',
 'France': 'Q142',
 'St. Augustine': 'Q487988',
 'Palma': 'Q8826',
 'Niza': 'Q33959',
 'Albania': 'Q222',
 'Rio de Janeiro': 'Q8678',
 'Janeiro': 'Q8678',
 'Puerto Rico': 'Q842360',
 'Zuloaga': 'Q12161577',
 'Gracias': 'Q2561339',
 'Buenos Aires': 'Q1486',
 'Provence': 'Q101081',
 'Rapallo': 'Q3480608',
 'Barcelona': 'Q1492',
 'United States': 'Q30',
 'Venezuela': 'Q717',
 'Doha': 'Q3861',
 'Ny': 'Q60',
 'Cherburgo': 'Q3667188',
 'New York': 'Q60',
 'Berlin': 'Q64',
 'Lisboa': 'Q597',
 'PeTROpOLis}': 'Q189043',
 'Buenos AIRES': 'Q1486',
 'CaLiFoRNIA': 'Q2163769',
 'NY': 'Q60',
 'Cruz': 'Q20143705',
 'San Agustin': 'Q487988',
 'Santa Teresa': 'Q1390081',
 'San Fernando': 'Q40584',
 'Aust

## Persons

In [305]:
keys = dict()

i = 0
for author in enumerate(persons['author'].unique().tolist()):
    keys[author[1]] = f"AUTH00{i+1}"
    i+=1
    
keys

{'mistral': 'AUTH001',
 'vo_to_sister_extra': 'AUTH002',
 'camus': 'AUTH003',
 'woolf': 'AUTH004',
 'joygasset': 'AUTH005'}

In [306]:
openrefine = pd.read_csv('openrefine_person.csv')

for i,row in openrefine.iterrows():
    if not(pd.isna(row['wikidata'])):
        keys[row['name']] = row['wikidata']

In [307]:

def wikidata_query(query):
    url = 'https://query.wikidata.org/sparql'
    try:
        r = requests.get(url, params = {'format': 'json', 'query': query})
        return r.json()['results']['bindings']
    except json.JSONDecodeError as e:
        raise Exception('Invalid query')

keys_list = list(keys.values())
keys_list = [f'wd:{key}' for key in keys_list]

wd_labels = dict()
    

item2label = wikidata_query('''
    SELECT ?item ?itemLabel WHERE {
    VALUES ?item { %s }
    SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}''' % " ".join(keys_list))

for result in item2label :
    item = re.sub(r".*[#/\\]", "", result['item']['value'])
    label = result['itemLabel']['value']
    wd_labels[item] = label
    


In [308]:
edges = []
nodes = []

for i,row in persons.iterrows():
    if row['value'] in keys:
        edges.append( [ keys[row['value']],keys[row['author']] ] )
        
        new_node = [ row['value'] , wd_labels[keys[row['value']]]]
        if new_node not in nodes:
            nodes.append(new_node)
        new_node = [ keys[row['author']] , row['author'] ]
        if new_node not in nodes:
            nodes.append(new_node)
            

In [309]:
with open('persons_edges.csv','w') as out:
    out.write('Source,Target\n')
    for edge in edges:
        out.write( f"'{edge[1]}','{edge[0]}'\n")
        
        
with open('persons_nodes.csv','w') as out:
    out.write('Id,Label\n')
    for node in nodes:
        if re.search('^AUTH',node[0]):
            out.write( f"'{node[0]}',{node[1]}\n")
        else:
            out.write( f"'{keys[node[0]]}',{node[1]}\n")